<a href="https://colab.research.google.com/github/boemer00/courses/blob/main/retrieval_augmented_qa_investment_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install libraries. Kernel restart required to downgrade a component library.
!pip -q install configparser langchain google-generativeai chromadb
!pip -q install transformers huggingface_hub sentence_transformers
!pip install 'google-cloud-aiplatform>=1.26.0'

In [ ]:
#@title Authentication for VertexAI.
!gcloud auth application-default login
# Use your project id here.
!gcloud config set project stellar-works-391900
!gcloud auth application-default set-quota-project stellar-works-391900

In [ ]:
#@title Useful libraries to import and play with.
import os
from dotenv import load_dotenv, find_dotenv # To load API keys from .env file.
# VertexAI also uses PalM but it's got more bells and whistles,
# e.g. you can send private data.
from langchain.llms import GooglePalm
# Many methods to transform text into embeddings.
from langchain.embeddings import GooglePalmEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings

# Prompting, LLMChain, Retrieval QA.
from langchain import PromptTemplate, LLMChain
from langchain.chains import RetrievalQA

# Retrieval and indexing of web data.
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.indexes import VectorstoreIndexCreator

# VertexAI
from langchain.llms import VertexAI
from IPython.display import display, Markdown, Latex


In [ ]:
#@title VertexAI: Complex multi-part question.
llm = VertexAI()
def build_llm_chain(llm):
  template = "Question: {question}\n\nAnswer: Let's think step by step.\n"
  prompt_open = PromptTemplate(template=template, input_variables=["question"])
  return LLMChain(prompt=prompt_open, llm=llm)


result = build_llm_chain(llm).run(
  "What NFL team won the Super Bowl in the year Justin Beiber was born?")
print(result)


In [ ]:
#@title VertexAI: code-bison to generate code.
def build_code_generator():
  code_bison_llm = VertexAI(model_name="code-bison")
  template = "Question: {question}\n\nAnswer: Write python code that answers the above question."
  prompt = PromptTemplate(template=template, input_variables=["question"])
  return LLMChain(prompt=prompt, llm=code_bison_llm)
question = "Write a python function that returns the nth fibonacci number?"
result = build_code_generator().run(question)
print(result)

In [ ]:
#@title Load some financial data for retrieval-augmentation.

def build_qa_bot(llm, urls):
  loader = WebBaseLoader(urls)
  index = VectorstoreIndexCreator(
          embedding=SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2"),
          text_splitter=RecursiveCharacterTextSplitter(
              chunk_size=1000,
              chunk_overlap=0,
              separators=[" ", ",", "\n"])).from_loaders([loader])

  return RetrievalQA.from_chain_type(llm=llm, chain_type="stuff",
                                      retriever=index.vectorstore.as_retriever(),
                                      input_key="question")

ask_fundrise = build_qa_bot(
    llm=llm,
    urls=["https://www.schroders.com/en/global/individual/insights/quarterly-markets-review-q1-2023/", 'https://fundrise.com/education/q1-2023-performance-update', ])



In [ ]:
#@title Retrieval-augmented QA

ask_fundrise("What is the outlook for Q1 2023?")

In [ ]:
ask_fundrise("Where should I invest my money?")

In [ ]:
ask_fundrise("What is a reason to invest in something other than equities?")